# IARA-Imagem em ESCALA — Resonance Forest no Colab

Escala a pesquisa de `make-shorts-video/iara_pinboard` (113 marcos) que já **bate o JPEG**, mas estava
faminta: treinava com **<1000 imagens** e **5000 passos** numa GPU fraca (6-8h).

Aqui: **~5000 imagens diversas** (10× mais) + treino longo + curva R-D completa vs JPEG.

**A arquitetura (fiel ao M45/M101/M113):**
- **PyramidN** — pirâmide RESIDUAL de N níveis (grosso→fino). Cada nível come o resíduo do anterior.
- **Entropia aprendida** (`logb`) — o custo em bits é otimizado junto (lei taxa-distorção).
- **Domínios EMERGEM** — clustering por cor+frequência, sem rótulo.
- **Sementes/galhos** — adapter por domínio no nível de alta-frequência.

**Antes de rodar:** `Ambiente de execução → Alterar tipo → GPU (T4)` → `Executar tudo`.


## 1) GPU + libs


In [ ]:
import torch, time
assert torch.cuda.is_available(), "Sem GPU! Ambiente de execução > Alterar tipo > GPU"
print("GPU:", torch.cuda.get_device_name(0))
T0=time.time()
def log(m): print(f"[{time.time()-T0:6.0f}s] {m}", flush=True)


## 2) Dataset em escala (COCO val2017 — 5000 imagens naturais diversas, ~1GB)

O projeto usava 451 imagens (DIV2K + frames de YouTube). Aqui **~5000 imagens diversas** —
diversidade real é o que faz os domínios emergirem de verdade.


In [ ]:
import os, glob, subprocess
DATA="/content/val2017"
if not os.path.isdir(DATA):
    print("baixando COCO val2017 (~780MB, 5000 fotos naturais diversas) …", flush=True)
    subprocess.run(["wget","-q","http://images.cocodataset.org/zips/val2017.zip","-O","/content/coco.zip"],check=True)
    subprocess.run(["unzip","-q","/content/coco.zip","-d","/content/"],check=True)
    os.remove("/content/coco.zip")
FILES=sorted(glob.glob(f"{DATA}/*.jpg"))
assert FILES, "dataset não baixou — rode a célula de novo"
print(f"{len(FILES)} imagens disponíveis  (o projeto original treinava com 451 no total)")


## 3) A arquitetura PyramidN (fiel ao marco45_deeppyramid.py)


In [ ]:
import numpy as np, torch.nn as nn, torch.nn.functional as F
LN2=float(np.log(2)); S=128; dev="cuda"
def chans(d,c): return [3]+[32,48,64,96,128][:d-1]+[c]
def enc_stack(cs):
    L=[]
    for i in range(len(cs)-1): L+=[nn.Conv2d(cs[i],cs[i+1],4,2,1), nn.GELU()]
    return nn.Sequential(*L[:-1])
def dec_stack(cs):
    L=[]
    for i in range(len(cs)-1): L+=[nn.ConvTranspose2d(cs[i],cs[i+1],4,2,1), nn.GELU()]
    return nn.Sequential(*L[:-1])
class PyramidN(nn.Module):
    """Pirâmide RESIDUAL: nível i codifica o que o nível i-1 não explicou (grosso→fino)."""
    def __init__(self, specs):
        super().__init__(); self.specs=specs
        self.encs=nn.ModuleList([enc_stack(chans(d,c)) for d,c in specs])
        self.decs=nn.ModuleList([dec_stack(list(reversed(chans(d,c)))) for d,c in specs])
        self.logb=nn.ParameterList([nn.Parameter(torch.zeros(c)) for _,c in specs])
    def _q(self,z,train): return z+(torch.rand_like(z)-0.5) if train else torch.round(z)
    def forward(self,x,train=True):
        rec=None; zs=[]; recs=[]
        for i in range(len(self.specs)):
            tgt = x if rec is None else (x-rec)          # <- come o RESÍDUO
            z=self._q(self.encs[i](tgt),train); zs.append(z)
            out=self.decs[i](z)
            rec = torch.sigmoid(out) if i==0 else (rec+out).clamp(0,1)
            recs.append(rec)
        return {"rec":rec,"recs":recs,"z":zs}
def rate(z,logb):   # custo em BITS (entropia laplaciana aprendida)
    b=F.softplus(logb)[None,:,None,None]+1e-3
    return (torch.log(2*b)+z.abs()/b).mean()/LN2
def kb_of(z):       # tamanho real medido (entropia empírica)
    v=z.detach().cpu().numpy().ravel().astype(np.int32)
    if v.size==0: return 0.0
    p=np.bincount(v-v.min()).astype(np.float64); p=p[p>0]/v.size
    return v.size*float(-(p*np.log2(p)).sum())/8/1024
def psnr(a,b):
    m=float(((a-b)**2).mean()); return 99.0 if m<1e-12 else 10*np.log10(1.0/m)
SPECS=[(5,24),(4,16),(3,12),(2,8)]   # coarse -> fine (do M113)
MULT=2**max(d for d,_ in SPECS)      # a pirâmide reduz 2^5=32× → lados devem ser múltiplos de MULT
print("PyramidN pronta ·",SPECS,"· lados devem ser múltiplos de",MULT)



## 4) Carregar as imagens (escala!)


In [ ]:
from PIL import Image
N_IMGS = 1200      # sobe se quiser (RAM do Colab aguenta ~2000 a 128px)
HELD   = 60        # held-out (nunca vistas no treino)
def load(fs):
    out=[]
    for f in fs:
        try:
            im=Image.open(f).convert("RGB")
            if min(im.size)<S: continue
            if min(im.size)>320: r=320/min(im.size); im=im.resize((int(im.width*r),int(im.height*r)))
            out.append(torch.from_numpy(np.asarray(im)).float().permute(2,0,1)/255.)
        except Exception: pass
    return out
ALL=load(FILES[:N_IMGS]); pool=ALL[:-HELD]; held=ALL[-HELD:]
log(f"treino {len(pool)} imgs · held-out {len(held)}  (o projeto original tinha 451 no total)")


## 5) Domínios EMERGEM (cor + frequência, sem rótulo) — fiel ao M101/M113


In [ ]:
def feat(img):
    x=F.interpolate(img[None],size=(S,S),mode="bilinear")[0]
    hist=torch.cat([torch.histc(x[c],bins=4,min=0,max=1) for c in range(3)])/(S*S)
    g=x.mean(0); fft=torch.fft.rfft2(g).abs(); h,w=fft.shape
    yy,xx=torch.meshgrid(torch.arange(h),torch.arange(w),indexing="ij")
    r=torch.sqrt((yy/h)**2+(xx/w)**2)
    bands=torch.tensor([fft[(r>=a)&(r<b)].mean() for a,b in [(0,.05),(.05,.15),(.15,.3),(.3,.5),(.5,1.)]])
    return torch.cat([hist,bands/(bands.sum()+1e-9)]).numpy()
Ftr=np.stack([feat(i) for i in pool]); mu,sd=Ftr.mean(0),Ftr.std(0)+1e-9; Ftr=(Ftr-mu)/sd
K=4; rng=np.random.default_rng(0); cent=Ftr[rng.choice(len(Ftr),K,replace=False)].copy()
for _ in range(50):
    lab=((Ftr[:,None]-cent[None])**2).sum(-1).argmin(1)
    for k in range(K):
        if (lab==k).any(): cent[k]=Ftr[lab==k].mean(0)
from collections import Counter
log(f"domínios emergentes: {dict(Counter(lab.tolist()))}")
import matplotlib.pyplot as plt
fig,ax=plt.subplots(K,6,figsize=(13,2.2*K))
for k in range(K):
    idx=[i for i in range(len(pool)) if lab[i]==k][:6]
    for j,i in enumerate(idx): ax[k,j].imshow(pool[i].permute(1,2,0)); ax[k,j].axis("off")
    for j in range(len(idx),6): ax[k,j].axis("off")
    ax[k,0].set_title(f"domínio {k} (n={int((lab==k).sum())})",loc="left",fontsize=9)
plt.tight_layout(); plt.show()


## 6) Treinar o NÚCLEO (longo) — o loss deve cair


In [ ]:
STEPS=8000; B=16; LAM=1.2e-3
def crops(idxs,n):
    out=[]
    for _ in range(n):
        im=pool[idxs[int(torch.randint(0,len(idxs),(1,)))]]
        i=int(torch.randint(0,im.shape[1]-S+1,(1,))); j=int(torch.randint(0,im.shape[2]-S+1,(1,)))
        out.append(im[:,i:i+S,j:j+S])
    return torch.stack(out).to(dev)
base=PyramidN(SPECS).to(dev); opt=torch.optim.AdamW(base.parameters(),lr=6e-4)
allidx=list(range(len(pool))); hist=[]
for step in range(STEPS):
    x=crops(allidx,B); out=base(x,train=True)
    # LOSS PROGRESSIVO (fiel ao M44, o marco que BATEU o JPEG): CADA grau é supervisionado.
    # Sem isto o modelo mata os graus grossos e joga tudo no último → a pirâmide COLAPSA.
    dist=sum(F.mse_loss(r,x) for r in out["recs"])
    loss=dist+LAM*sum(rate(z,lb) for z,lb in zip(out["z"],base.logb))
    opt.zero_grad(); loss.backward(); opt.step(); hist.append(loss.item())
    if step%500==0:
        with torch.no_grad(): kbs=[kb_of(z) for z in base(x,train=False)["z"]]
        log(f"núcleo {step}: loss {np.mean(hist[-100:]):.4f} | KB por grau {[round(k,2) for k in kbs]}  <- todos >0 = pirâmide VIVA")
torch.save(base.state_dict(),"/content/nucleo.pt"); base.eval()
plt.figure(figsize=(6,3)); plt.plot(np.convolve(hist,np.ones(100)/100,"valid")); plt.title("loss do núcleo"); plt.show()




## 7) 🏆 A CURVA R-D vs JPEG (o número que importa)

Cada nível da pirâmide = um ponto de qualidade. Comparamos com JPEG **na mesma banda**.


In [ ]:
import io
_JC={}
def jpeg_curve(img_t, key):
    """varre a curva JPEG UMA vez por imagem (cacheada) → [(kb,psnr)]"""
    if key in _JC: return _JC[key]
    arr=(img_t.permute(1,2,0).numpy()*255).astype(np.uint8); ref=arr.astype(np.float32)/255; pts=[]
    for q in range(2,96,2):
        b=io.BytesIO(); Image.fromarray(arr).save(b,"JPEG",quality=q)
        rec=np.asarray(Image.open(io.BytesIO(b.getvalue())).convert("RGB"),np.float32)/255
        pts.append((len(b.getvalue())/1024, psnr(rec,ref)))
    _JC[key]=pts; return pts
def jpeg_at(img_t, kb_target, key):
    """PSNR do JPEG na MESMA banda (KB) — o baseline honesto"""
    pts=jpeg_curve(img_t,key); kb,p=min(pts,key=lambda t:abs(t[0]-kb_target)); return p,kb
rows=[]
with torch.no_grad():
    for L in range(1,len(SPECS)+1):                      # níveis progressivos = pontos da curva
        ps,kbs,pj=[],[],[]
        for i,im in enumerate(held[:24]):
            x=im[:, :im.shape[1]//MULT*MULT, :im.shape[2]//MULT*MULT][None].to(dev)
            o=base(x,train=False)
            rec=o["recs"][L-1][0].cpu(); kb=sum(kb_of(z) for z in o["z"][:L])
            ps.append(psnr(rec.numpy(),x[0].cpu().numpy())); kbs.append(kb)
            pj.append(jpeg_at(x[0].cpu(),kb,i)[0])
        rows.append((L,float(np.mean(kbs)),float(np.mean(ps)),float(np.mean(pj))))
        log(f"nível {L}: {np.mean(kbs):5.2f} KB · IARA {np.mean(ps):5.2f} dB · JPEG {np.mean(pj):5.2f} dB · Δ {np.mean(ps)-np.mean(pj):+.2f}")
plt.figure(figsize=(6,4))
plt.plot([r[1] for r in rows],[r[2] for r in rows],"o-",label="IARA pirâmide")
plt.plot([r[1] for r in rows],[r[3] for r in rows],"s--",label="JPEG")
plt.xlabel("KB"); plt.ylabel("PSNR (dB)"); plt.legend(); plt.grid(alpha=.3); plt.title("curva R-D — acima = melhor"); plt.show()
print("Δ médio vs JPEG:", round(float(np.mean([r[2]-r[3] for r in rows])),2), "dB")



## 8) Ver a reconstrução progressiva (os PASSOS: grosso → fino)


In [ ]:
with torch.no_grad():
    im=held[0]; x=im[:, :im.shape[1]//MULT*MULT, :im.shape[2]//MULT*MULT][None].to(dev)
    o=base(x,train=False)
    fig,ax=plt.subplots(1,len(SPECS)+1,figsize=(3.2*(len(SPECS)+1),3.4))
    ax[0].imshow(x[0].cpu().permute(1,2,0)); ax[0].set_title("original"); ax[0].axis("off")
    for L in range(len(SPECS)):
        r=o["recs"][L][0].cpu(); kb=sum(kb_of(z) for z in o["z"][:L+1])
        ax[L+1].imshow(r.permute(1,2,0).clamp(0,1)); ax[L+1].axis("off")
        ax[L+1].set_title(f"grau {L}: {psnr(r.numpy(),x[0].cpu().numpy()):.1f} dB / {kb:.1f} KB",fontsize=9)
    plt.tight_layout(); plt.show()



## 9) Sementes/galhos: adapter por domínio (paga ou não paga?)

Fiel ao M101/M104: adapter no nível de **alta-frequência** por domínio. Testa **certo vs errado**
(se o errado piorar, a especialização é real).


In [ ]:
import copy
ADAPT=1500
adapters={}
for k in range(K):
    idx=[i for i in range(len(pool)) if lab[i]==k]
    if len(idx)<30: continue
    ea=copy.deepcopy(base.encs[-1]); da=copy.deepcopy(base.decs[-1])
    for p in list(ea.parameters())+list(da.parameters()): p.requires_grad_(True)
    oa=torch.optim.AdamW(list(ea.parameters())+list(da.parameters()),lr=4e-4)
    e0,d0=base.encs[-1],base.decs[-1]; base.encs[-1],base.decs[-1]=ea,da; base.train()
    for s in range(ADAPT):
        x=crops(idx,B); o=base(x,train=True)
        l=sum(F.mse_loss(r,x) for r in o["recs"])+LAM*rate(o["z"][-1],base.logb[-1])
        oa.zero_grad(); l.backward(); oa.step()
    base.encs[-1],base.decs[-1]=e0,d0; base.eval()
    adapters[k]=(ea.eval(),da.eval())
    nb=sum(p.numel() for p in ea.parameters())+sum(p.numel() for p in da.parameters())
    log(f"adapter dom{k}: {len(idx)} imgs · {nb} params · fp16 {nb*2/1024:.0f}KB")
# validação: núcleo vs adapter CERTO vs adapter ERRADO (held-out)
Fte=(np.stack([feat(i) for i in held])-mu)/sd
hlab=((Fte[:,None]-cent[None])**2).sum(-1).argmin(1)
def ev(im,ad=None):
    with torch.no_grad():
        x=im[:, :im.shape[1]//MULT*MULT, :im.shape[2]//MULT*MULT][None].to(dev)
        if ad is not None:
            e0,d0=base.encs[-1],base.decs[-1]; base.encs[-1],base.decs[-1]=ad
        r=base(x,train=False)["rec"][0].cpu()
        if ad is not None: base.encs[-1],base.decs[-1]=e0,d0
        return psnr(r.numpy(),x[0].cpu().numpy())
for k in sorted(adapters):
    ids=[i for i in range(len(held)) if hlab[i]==k][:8]
    if not ids: continue
    others=[j for j in adapters if j!=k]
    b_=np.mean([ev(held[i]) for i in ids]); c_=np.mean([ev(held[i],adapters[k]) for i in ids])
    msg=f"dom{k}: núcleo {b_:.2f} · adapter CERTO {c_:.2f} ({c_-b_:+.2f})"
    if others:
        w_=np.mean([ev(held[i],adapters[others[0]]) for i in ids]); msg+=f" · adapter ERRADO {w_:.2f} ({w_-b_:+.2f})"
    print(msg)




## 10) Onde ir depois (o que o Colab ainda destrava)
- **Mais dados**: sobe `N_IMGS` (2000+) e `STEPS` (20000) → descobre se o platô era fome.
- **Mais níveis**: acrescenta `(6,32)` no início do `SPECS` = detalhe mais fino.
- **Vídeo**: I-frame + delta temporal com motion compensation (M85-M88 já deram −32% vs JPEG).
- **Quantizar os galhos**: int6/int4 nos adapters (M106 = 85-90% do ganho com metade dos bytes).
- **Casar com o resto da IARA**: água÷grau entre os graus, verificador decidindo quando descer.
